In [1]:

##############################################################
###  This Notebook can do interact, see instructions below ###
##############################################################




# -*- coding: utf-8 -*-
"""
Created on Mon Jun 30 22:06:32 2025

@author: jiaxx213
"""
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import os
import sys
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split, Subset

from functions_basic import var_list, load_sourcedata, exist_split, create_split, random_new_seed, get_all_seeds
from functions_data import Data_Manager
from functions_LITT import Trainer_LITT
from functions_Events import Event_Analyzer
from functions_Path import Path_Analyzer

#method_name = "T2" # 
split_rate = 0.7
hidden_size = 16

n_batch = 32
n_epochs = 35
n_epoches_gamma = 35

#-------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU is available!")
    print(torch.cuda.get_device_name(0))
else:
    print("Warning! No GPU available, back to CPU")

#-------------------
var_dict = {s: i for i, s in enumerate(var_list)}
dim_x = len(var_list)

#'HF','IS','ARR','CA'
oc_names = ['HF','IS','ARR','AM','CA']



###### Comment Code of loading data and training ######
###### The current version is only for result display, without data loading ######

# # ---------------------
# loaded_data = load_sourcedata()
# sequences = loaded_data['sequences']
# time_sequences = loaded_data['time_sequences']
# targets_dict = loaded_data['targets']
# last_days = loaded_data['last_days']
# pid_list = loaded_data['pid_list']

# oc_targets = {k: targets_dict[k] for k in oc_names}
# last_days = torch.tensor(last_days, dtype=torch.float32)

# # ---------------------
# data_manager = Data_Manager(sequences, time_sequences, oc_targets, hidden_size, n_batch, device)
# data_manager.load_full()

#trainer_LITT = Trainer_LITT(data_manager)
#trainer_LITT.loading()

#dataloader_LITT, _ = data_manager.get_DataLoader('LITT')

#trainer_LITT.run_training(n_epochs, dataloader_LITT)

#trainer_LITT.run_flipping(n_epochs, data_manager)


Warning! No GPU available, back to CPU


In [2]:
# event_analyzer = Event_Analyzer(data_manager)
# event_analyzer.load_events()

# recur_dict = event_analyzer.do_analyze(log_out=False)
# event_path_fname = './BC_LITT/full_LITT_event_paths'
# with open(event_path_fname, 'rb') as f:
#     recur_dict = pickle.load(f)


# slt_evts = ['metformin_1', 'insulin_1', 'sulfonylurea_1', 'glp1_recpt_1', 'sglt2_inhib_1', 'ezetimibe_1', 'fibrates_1', 'bile_acid_1', 'diuretics_1', 'diabetes_1', 'hypertension_1', 'obesity_1', 'smoking_1', 'ckd_1']
# event_analyzer.disp_3heatmap([("radiation",1)], [("anthracyclins",1)], [("targeted",1)], slt_evts)

# path_analyzer = Path_Analyzer(data_manager)
# path_analyzer.loading()

In [3]:


event_analyzer = Event_Analyzer()
event_analyzer.load_events()

event_path_fname = './full_LITT_event_paths'
with open(event_path_fname, 'rb') as f:
    recur_dict = pickle.load(f)


#path_analyzer = Path_Analyzer(data_manager)
path_analyzer = Path_Analyzer()
path_analyzer.loading()

In [4]:
#slt_keys = ['anthracyclins', 'targeted', 'radiation']
slt_keys = ['anthracyclins', 'targeted', 'radiation', 'diabetes', 'hypertension', 'obesity', 'ckd', 'metformin', 'insulin', 'sulfonylurea', 'glp1_recpt', 'sglt2_inhib', 'ezetimibe', 'fibrates', 'bile_acid', 'diuretics']

path_list = path_analyzer.select_path(recur_dict, slt_keys, 6)

path_pool = path_analyzer.compact_path([1.0, 2, 0.05, 2], [])

###### Even Display Names and Explanation ######
# 'anthracyclins':'Chemo' # Chemotherapy
# 'targeted':'Targt' # Target theropy
# 'radiation':'Radia' # Radiation procedure
# 'metformin':'M_Mtf','insulin':'M_Ins',  # medicine metfomin and insulin
# 'sulfonylurea':'M_Sul', # second line medicine after insulin
# 'sglt2_inhib':'M_SGL',  # medicine to benifit cardiovascular but pressure to kidney
# 'glp1_recpt':'M_GLP', # medicine to mimic the natural hormone GLP-1, stimulating insulin release
# 'ezetimibe': 'M_Ezt', # medicine to inhibit cholesterol absorption in intestine; reduce cardiovascular risk)
# 'pcsk9_inhib':'M_PCS', # medicine to lower LDL "bad" cholesterol, well-tolerated, limited side effect
# 'fibrates':'M_Fib', # medicine to raise "good" HDL cholesterol; reduce heart disease risk
# 'bile_acid':'M_BAS', # medicine to lower bad LDL cholesterol by binding to bile acids
# 'niacin':'M_VB3', # essential water-soluble nutrient; raising "good" HDL and lowering "bad" LDL
# 'diuretics':'M_Diu', # Water pill; help kidneys, treat hert failure
# 'hyperlipidemia':'Dx_HL', 'diabetes':'Dx_DB', 'hypertension':'Dx_HT', 'obesity':'Dx_Ob', 'smoking':'DxSmk', 'ckd':'DxCKD' # Diagnosis as named


###### Interact Instruction ######
# 1. Edit slt_keys to change the candidate events you want to include in the path exploration. Their displayed name matching and interpretation are provided above
# 2. The first parameter of path_analyzer.compact_path() is the tolerance thresholds for considering two events as equal when merging the ends of paths for a better exhibition
#    e.g., [1.0, 2, 0.05, 2] means: kurt_tol=1.0 to allow 1.0 diff on kurt value; cohort_tol=2 to allow 2 diff when counting the major cohort that used to calculate kurt value
#                                   rate_tol=0.05 to allow 5% diff on positive rates; true_cohort_tol=2 to all 2 diff when counting the cohort truely following this path
#    e.g., [] means: leaving it as empty for exact value comparition with no tolerance
# 3. The second parameter of path_analyzer.compact_path() is the prefix events of paths you want to display, that is, to specify a header
#    e.g., ['Chemo_1', 'Targt_1'] means: Only display the paths in header  Chemo_1 ───> Targt_1 ───> ...


###### To Confirm the Effectiveness ######
# The display format: Event_Name (kurt_value; true_cohort_size / model_selected_cohort_size; positive_rate)
# The effectiveness reflects in three aspects:
#     1) Major of the model-selected cohort appear to truly follow the path, i.e., the / split two cohort size numbers are close
#     2) Most of the paths show a clear increasing (or decreasing) trend in positive rates, which can identify the trojectories of patients getting more serious (or more health)
#     3) The size of truly-following cohort is positively correlated with the model-derived kurt value, demonstrating that LITT can evaluate the significance of events in trojectories


n_path = 165
n_cluster = 33
==== cluster #0 ==== 

[ 14]:                            ┌───> Targt_2( 27.08;  83/  85; 42%)────────────────────────────────────>───┐ [ 11, 3]                   
[ 12]:                            │                                ┌───> Radia_1( 22.03;  29/  30; 53%)       │                            
[ 11]:                            │                                │                                ┌───> Radia_1( 20.32;  27/  28; 50%)   
[ 10]: Chemo_3( 39.49; 732/ 753; 36%)-> Targt_1( 27.40;  84/  86; 41%)-> Targt_2( 28.23;  81/  83; 43%)-> Radia_2( 21.08;  26/  27; 51%)   
[ 13]:                            └───> Targt_2( 27.08;  83/  85; 42%)────────────────────────────────────>───┘ [ 10, 3]                   
[ 15]:                            └───> M_Diu_1( 17.25; 148/ 151; 59%)-> Radia_1( 17.71;  23/  24; 45%)-> Radia_2( 14.78;  20/  21; 38%)   
[ 16]:                            │                                │                                └───> Rad